# Naive Baseline

## 1. Data Imports and Exploration

### 1.a. Imports

In [1]:
import signals
import backtester 
import backtester_realistic
import returns
import market_env_label
import market_env_analysis
import metrics
import pandas as pd
import numpy as np
import load_data as ld
import os 
import glob
from pprint import pprint

### 1.b. Load Data

In [14]:
new_data_loader = ld.DataLoader()
folder_path = "./CTA_data"
path_list = glob.glob(os.path.join(folder_path, "*.csv"))
new_data_loader.add_data(path_list)
price_data = new_data_loader.get_data_dictionary() 

In [15]:
print(price_data)

{'XLB_ohlcv_1d.csv':                        ts_event  rtype  publisher_id  instrument_id   open  \
0     2020-01-02 00:00:00+00:00     35            43          17674  61.90   
1     2020-01-03 00:00:00+00:00     35            43          17674  59.83   
2     2020-01-06 00:00:00+00:00     35            43          17674  59.55   
3     2020-01-07 00:00:00+00:00     35            43          17674  59.46   
4     2020-01-08 00:00:00+00:00     35            43          17674  59.56   
...                         ...    ...           ...            ...    ...   
1625  2026-06-23 00:00:00+00:00     35            43          17674  51.18   
1626  2026-06-24 00:00:00+00:00     35            43          17674  50.80   
1627  2026-06-25 00:00:00+00:00     35            43          17674  51.10   
1628  2026-06-26 00:00:00+00:00     35            43          17674  51.82   
1629  2026-06-29 00:00:00+00:00     35            43          17674  51.85   

        high     low  close   volume symbo

### 1.c. Data Exploration

In [22]:
sname = "AGG"
price_data_set = price_data[sname + "_ohlcv_1d.csv"]
price_data_set.head()

,ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
0,2020-01-02 00:00:00+00:00,35,43,429,112.45,112.97,112.400,112.97,1914638,AGG
1,2020-01-03 00:00:00+00:00,35,43,429,112.73,113.44,112.390,113.01,637221,AGG
2,2020-01-06 00:00:00+00:00,35,43,429,113.13,113.13,112.850,112.92,8190730,AGG
3,2020-01-07 00:00:00+00:00,35,43,429,112.93,112.93,112.780,112.80,699639,AGG
4,2020-01-08 00:00:00+00:00,35,43,429,112.85,112.93,112.545,112.61,1842584,AGG


In [23]:
prices = price_data_set['close']

In [24]:
returns_series = returns.get_returns_series(prices)

mean_return = returns_series.mean()
daily_vol = returns_series.std()

summary = {
    "mean return" : float(mean_return),
    "annual return" : float(mean_return * 252),
    "daily vol" : float(daily_vol),
    "annual vol" : float(daily_vol * np.sqrt(252)),
    "return range" : float(returns_series.max() - returns_series.min()),
    "max return" : float(returns_series.max()),
    "min return" : float(returns_series.min())
}

for name, val in summary.items():
    print(name+": ", val)

mean return:  -6.956049822441868e-05
annual return:  -0.017529245552553505
daily vol:  0.004327200627612019
annual vol:  0.06869218040246411
return range:  0.05713759256262907
max return:  0.02485035370941402
min return:  -0.03228723885321505


## 2. Naive Baseline Signals

### 2.a. Momentum Signal

In [7]:
# Example usage
price_data_set = returns.get_stock_data('SPY', '2025-07-05', '2026-07-05')
# prices = price_data_set[('Close', 'SPY')]
prices = returns.convert_to_raw_series('SPY', price_data_set)
returns_series = returns.get_returns_series(prices)
signal_mom = signals.momentum_signal(prices, window=10)

bt_mom = backtester.Backtester(prices, signal_mom)
bt_mom.run_backtest()
bt_mom.print_backtest_results()

[*********************100%***********************]  1 of 1 completed

                prices  signal  position      cash  asset_value  \
Date                                                              
2025-07-07  613.878052       0       0.0  100000.0          0.0   
2025-07-08  613.541809       0       0.0  100000.0          0.0   
2025-07-09  617.221008       0       0.0  100000.0          0.0   
2025-07-10  618.961731       0       0.0  100000.0          0.0   
2025-07-11  616.785828       0       0.0  100000.0          0.0   
2025-07-14  617.962830       0       0.0  100000.0          0.0   
2025-07-15  615.322083       0       0.0  100000.0          0.0   
2025-07-16  617.379272       0       0.0  100000.0          0.0   
2025-07-17  621.157349       0       0.0  100000.0          0.0   
2025-07-18  620.702454       0       0.0  100000.0          0.0   

            total_portfolio_value  daily_pnl  
Date                                          
2025-07-07               100000.0        0.0  
2025-07-08               100000.0        0.0  
2025-07

### 2.b. MACD Signal

In [8]:
signal_macd = signals.macd_signal(prices)

bt_macd = backtester.Backtester(prices, signal_macd)
bt_macd.run_backtest()
bt_macd.print_backtest_results()

                prices  signal  position           cash  asset_value  \
Date                                                                   
2025-07-07  613.878052       0       0.0  100000.000000     0.000000   
2025-07-08  613.541809      -1       0.0  100000.000000     0.000000   
2025-07-09  617.221008       1      -1.0  100617.221008  -617.221008   
2025-07-10  618.961731       1       1.0   99379.297546   618.961731   
2025-07-11  616.785828       1       1.0   99379.297546   616.785828   
2025-07-14  617.962830       1       1.0   99379.297546   617.962830   
2025-07-15  615.322083       1       1.0   99379.297546   615.322083   
2025-07-16  617.379272       1       1.0   99379.297546   617.379272   
2025-07-17  621.157349       1       1.0   99379.297546   621.157349   
2025-07-18  620.702454       1       1.0   99379.297546   620.702454   

            total_portfolio_value  daily_pnl  
Date                                          
2025-07-07          100000.000000   0.000

## 3. More Realistic Backtest

### 3.a. Momentum Signal

In [9]:
bt_mom = backtester_realistic.RealisticBacktester(prices, signal_mom, initial_cash=10000, cost_per_trade=10, slippage=0.01)
bt_mom.run_backtest()
bt_mom.print_backtest_results()

                prices  signal  position     cash  asset_value  \
Date                                                             
2025-07-07  613.878052       0       0.0  10000.0          0.0   
2025-07-08  613.541809       0       0.0  10000.0          0.0   
2025-07-09  617.221008       0       0.0  10000.0          0.0   
2025-07-10  618.961731       0       0.0  10000.0          0.0   
2025-07-11  616.785828       0       0.0  10000.0          0.0   
2025-07-14  617.962830       0       0.0  10000.0          0.0   
2025-07-15  615.322083       0       0.0  10000.0          0.0   
2025-07-16  617.379272       0       0.0  10000.0          0.0   
2025-07-17  621.157349       0       0.0  10000.0          0.0   
2025-07-18  620.702454       0       0.0  10000.0          0.0   

            total_portfolio_value  daily_pnl  
Date                                          
2025-07-07                10000.0        0.0  
2025-07-08                10000.0        0.0  
2025-07-09         

### 3.b. MACD Signal

In [10]:
bt_macd = backtester_realistic.RealisticBacktester(prices, signal_macd, initial_cash=10000, cost_per_trade=10, slippage=0.01)
bt_macd.run_backtest()
bt_macd.print_backtest_results()

                prices  signal  position          cash  asset_value  \
Date                                                                  
2025-07-07  613.878052       0       0.0  10000.000000     0.000000   
2025-07-08  613.541809      -1       0.0  10000.000000     0.000000   
2025-07-09  617.221008       1      -1.0  10601.048798  -617.221008   
2025-07-10  618.961731       1       1.0   9330.746102   618.961731   
2025-07-11  616.785828       1       1.0   9330.746102   616.785828   
2025-07-14  617.962830       1       1.0   9330.746102   617.962830   
2025-07-15  615.322083       1       1.0   9330.746102   615.322083   
2025-07-16  617.379272       1       1.0   9330.746102   617.379272   
2025-07-17  621.157349       1       1.0   9330.746102   621.157349   
2025-07-18  620.702454       1       1.0   9330.746102   620.702454   

            total_portfolio_value  daily_pnl  
Date                                          
2025-07-07           10000.000000   0.000000  
2025-0

## 4. Market Environment Analysis

In [11]:
rolling_volatility = market_env_label.get_rolling_volatility(bt_mom.table["total_portfolio_value"], window=20)
bt_mom.table["volatility_classification"] = market_env_label.get_volatility_classification(rolling_volatility)
rolling_volume = market_env_label.get_rolling_volume(bt_mom.table["total_portfolio_value"], window=20)
bt_mom.table["volume_classification"] = market_env_label.get_volume_classification(rolling_volume)

### 4.a. Filter by Volume

In [12]:
summary = market_env_analysis.analyze_market_environment(bt_mom.table, group_by="volatility_classification")

In [13]:
pprint(summary)

{'high volatility': {'annual_sharpe': np.float64(-5.985117290040533),
                     'annual_volatility': np.float64(0.07652437572732268),
                     'average_turnover': np.float64(0.45614035087719296),
                     'max_drawdown': np.float64(-0.09919237842796702),
                     'total_return': np.float64(-0.03543371381244098),
                     'total_turnover': np.float64(26.0),
                     'trade_frequency': np.float64(0.4482758620689655)},
 'low volatility': {'annual_sharpe': np.float64(-4.516371533142596),
                    'annual_volatility': np.float64(0.09831755779492207),
                    'average_turnover': np.float64(0.24561403508771928),
                    'max_drawdown': np.float64(-0.1011212116073057),
                    'total_return': np.float64(-0.1499439479335536),
                    'total_turnover': np.float64(14.0),
                    'trade_frequency': np.float64(0.2413793103448276)},
 'mid volatility': {'annual